# Phase 1 — build the dataset

Ingest source datasets → manifest → grouped split → verify → stats → YOLO export.

The manifest is the source of truth; the YOLO tree is regenerated from it. Everything here is a thin call into `alpr.data` — the logic is tested in CI, not in this notebook.

**Detection is trained region-agnostically.** A plate detector mostly learns "small bright rectangle on a vehicle" and transfers across countries; the India/Germany split lives in Phase 5's grammar validators, which need no data at all. That is what lets us use whatever is openly licensed.

In [ ]:
REPO = "https://github.com/fayazhussain2821/ALPR.git"
BRANCH = "phase-1/dataset"

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $BRANCH && git pull --quiet
else:
    !git clone --quiet --branch $BRANCH $REPO /content/ALPR

%cd /content/ALPR
!pip install --quiet -e .
!git log --oneline -1

## 1. Acquire sources

Each source is downloaded into its own directory under `data/raw/<name>/`. Fill in the download for each — Roboflow needs an API key, Kaggle needs `kaggle.json`, UC3M-LP is a direct Zenodo download.

**Licences differ and they matter**, because Phase 1 publishes a *derived* dataset:

| Source | Licence | Redistributable? |
|---|---|---|
| UC3M-LP | ODbL-1.0 | Yes, but the derived DB must also be ODbL |
| Roboflow US-EU | CC BY 4.0 | Yes, with attribution |
| DataCluster sample | CC BY-NC-ND 4.0 | **No** — NoDerivatives forbids publishing a re-split |

Tag every ingest with `source=` so a non-redistributable set can be filtered out before upload.

In [ ]:
import os
from pathlib import Path

from alpr.env import setup_api_keys

# Reads KAGGLE_API_TOKEN and ROBOFLOW_API_KEY from Colab's Secrets panel (the
# 🔑 icon in the left sidebar) and puts them in the environment for the client
# libraries below. Raises with instructions if either is not configured.
#
# NEVER paste a key into this cell. Notebook cells are committed to git, this
# repo is public, and GitHub keeps pushed objects reachable long after a
# commit is amended away — a leaked key there has to be rotated, not deleted.
setup_api_keys()

RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

# --- UC3M-LP (ODbL-1.0) -------------------------------------------------
# Zenodo record 17152029; ships labels2yolo.py to convert to YOLO format.
# !wget -q -O uc3m.zip https://zenodo.org/records/17152029/files/<file>.zip
# !unzip -q uc3m.zip -d {RAW}/uc3m-lp

# --- Roboflow (CC BY 4.0) -----------------------------------------------
# Roboflow exports as train/valid/test subdirs, each with images/ + labels/,
# so ingest each split directory separately rather than a single images/ path.
# !pip install --quiet roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
# rf.workspace("<workspace>").project("<project>").version(1).download(
#     "yolov8", location=str(RAW / "roboflow-us-eu")
# )

# --- Kaggle -------------------------------------------------------------
# Check the licence on the dataset page before ingesting: a CC BY-NC-ND set
# cannot be redistributed as part of our derived dataset.
# !pip install --quiet kagglehub
# import kagglehub
# path = kagglehub.dataset_download("<owner>/<dataset-slug>")

!ls -R {RAW} | head -40

## 2. Ingest into one manifest

`from_yolo_dir` collapses source class ids to a single `license_plate` class and reports what it skipped, so a shrinking training set is visible rather than silent.

In [ ]:
from alpr.data import Region, from_yolo_dir, write_manifest

SOURCES = [
    # (images dir, source tag, region, keep_classes, id_prefix)
    # (RAW / "uc3m-lp/images",              "uc3m-lp",        Region.UNKNOWN, None,           "uc3m-"),
    # (RAW / "roboflow-us-eu/train/images", "roboflow-us-eu", Region.UNKNOWN, frozenset({0}), "rfus-"),
]

records = []
for images_dir, source, region, keep, prefix in SOURCES:
    report = from_yolo_dir(
        images_dir,
        source=source,
        region=region,
        keep_classes=keep,
        id_prefix=prefix,
        # Paths relative to the shared root so sources compose. Without this,
        # two sources both containing img001.jpg resolve to the same file.
        path_root=RAW,
        strict=False,  # record bad labels rather than aborting a long ingest
    )
    print(f"--- {source} ---")
    print(report.summary())
    records.extend(report.records)

print(f"\ntotal: {len(records)} images")
write_manifest(records, "data/manifest.jsonl")

## 3. Split, and prove it does not leak

Grouped (frames from one clip stay together), stratified by region, and deterministic from the seed. `verify_split` raises rather than warns — a bad split is worth stopping for, because everything downstream is measured against it.

In [ ]:
from alpr.data import read_manifest, split_records, verify_split

SEED = 0

records = read_manifest("data/manifest.jsonl")
assignment = split_records(records, seed=SEED)
verify_split(records, assignment)

print("split verified — no leakage, full coverage")
for split, count in sorted(assignment.counts(records).items()):
    print(f"  {split.value:<6} {count}")

## 4. Stats and the Phase 1 exit criteria

In [ ]:
from alpr.data import check_exit_criteria, compute_stats

stats = compute_stats(records, assignment)
print(stats.report())

failures = check_exit_criteria(stats)
print()
if failures:
    print("Phase 1 NOT met:")
    for f in failures:
        print(f"  - {f}")
else:
    print("Phase 1 exit criteria met.")

The `tiny (<32px wide)` figure is the one to watch. Those plates are unreadable in principle — the detector may find them but Phase 4's OCR has no strokes to work with. A high share caps end-to-end accuracy no matter how good the detector gets.

## 5. Export the YOLO tree for Phase 2

In [ ]:
from alpr.data import export_yolo

result = export_yolo(
    records,
    assignment,
    image_root="data/raw",
    out_root="data/yolo",
    symlink=True,  # copying tens of thousands of JPEGs wastes the session
)
print(result.summary())
print()
print(result.data_yaml.read_text())

## 6. Publish

Push the manifest and the split to the Hub so Phase 2 trains against a fixed, named dataset version rather than whatever happened to be in this session.

**Exclude any source whose licence forbids redistribution** (the CC BY-NC-ND one) before uploading images.

In [ ]:
REDISTRIBUTABLE = {"uc3m-lp", "roboflow-us-eu"}

publishable = [r for r in records if r.source in REDISTRIBUTABLE]
print(f"{len(publishable)} of {len(records)} records are redistributable")

# from huggingface_hub import HfApi
# write_manifest(publishable, "data/manifest_public.jsonl")
# HfApi().upload_file(
#     path_or_fileobj="data/manifest_public.jsonl",
#     path_in_repo="manifest.jsonl",
#     repo_id="Babblu2821/alpr-plates",
#     repo_type="dataset",
# )